In [1]:
import math
import numpy as np
from gurobipy import Model, GRB, quicksum
from geopy.distance import geodesic

# -----------------------
# Parametry wejściowe
# -----------------------

s = 2  # dostępne lokalizacje nowe
e = 1  # istniejące schrony
h = 2  # liczba obiektów mieszkalnych

P = [40, 4, 4, 40]  # kary – nie będą tu używane (to by miało znaczenie w QUBO)

c = [2, 2, 0]  # koszty budowy (0 dla istniejącego)
p = 4          # budżet
v = [2, 1, 1]  # pojemności schronów
K = 5
d = 1

# lokalizacje schronów i
L = [
    [51.103504, 17.086370],
    [51.100217, 17.082551],
    [51.100390, 17.099490]
]

# lokalizacje obiektów mieszkalnych n
M = [
    [51.103910, 17.086310],
    [51.105504, 17.186370]
]

# odległości r_{i,n} w kilometrach
r = [[geodesic(l, m).kilometers for m in M] for l in L]
r = np.array(r)

# -----------------------
# Model Gurobi
# -----------------------

model = Model("shelter_location")

x = {}
y = {}
z = {}

for i in range(s + e):
    y[i] = model.addVar(vtype=GRB.BINARY, name=f"y_{i}")
for n in range(h):
    z[n] = model.addVar(vtype=GRB.BINARY, name=f"z_{n}")
for i in range(s + e):
    for n in range(h):
        x[(i, n)] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{n}")

model.update()

# funkcja celu
obj = (
        quicksum(r[i, n] * x[(i, n)] for i in range(s + e) for n in range(h))
        + quicksum(c[i] * y[i] for i in range(s + e))
        + quicksum(K * z[n] for n in range(h))
)

model.setObjective(obj, GRB.MINIMIZE)

# ograniczenia – każdy obiekt przypisany dokładnie do 1 schronu
for n in range(h):
    model.addConstr(quicksum(x[(i, n)] for i in range(s + e)) + z[n] == 1, name=f"assign_{n}")

# ograniczenia – pojemności
for i in range(s + e):
    model.addConstr(quicksum(x[(i, n)] for n in range(h)) <= v[i] * y[i], name=f"cap_{i}")

# ograniczenie – maksymalna odległość
for i in range(s + e):
    for n in range(h):
        if r[i, n] > d:
            model.addConstr(x[(i, n)] == 0, name=f"dist_{i}_{n}")

# budżet
model.addConstr(quicksum(c[i] * y[i] for i in range(s + e)) <= p, name="budget")

for i in range(s, s + e):
    model.addConstr(y[i] == 1)

model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"\nOptymalny koszt całkowity = {model.objVal:.3f}\n")
    print("Schrony zbudowane / aktywne:")
    for i in range(s+e):
        if y[i].X > 0.5:
            print(f"  - schron {i} (koszt={c[i]}, capacity={v[i]})")
    print("\nPrzypisania obiektów:")
    for n in range(h):
        for i in range(s+e):
            if x[(i,n)].X > 0.5:
                print(f"  obiekt {n} -> schron {i} (dist={r[i,n]:.3f} km)")


Restricted license - for non-production use only - expires 2026-11-23
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-14450HX, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 7 rows, 9 columns and 18 nonzeros
Model fingerprint: 0x92b0cd32
Variable types: 0 continuous, 9 integer (9 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [5e-02, 7e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 9.0528823
Presolve removed 4 rows and 5 columns
Presolve time: 0.00s
Presolved: 3 rows, 4 columns, 7 nonzeros
Variable types: 0 continuous, 4 integer (4 binary)

Root relaxation: objective 8.157169e+00, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Inc